# EnergyDataModel — Quickstart

`energydatamodel` (EDM) is a typed, tree-shaped data model for describing energy systems — assets, areas, interconnections, and the time series attached to each. Every structural class ultimately inherits from a single `Element` root, with three sibling subtrees and one cross-cutting mixin:

- **`Node`** — graph vertices (equipment, areas, sensors, grid topology points). Has `members` and `tz`. Subclassed by `NodeAsset`, `GridNode`, `Sensor`, and `Area`.
- **`Edge`** — a directed relationship between two nodes (`Line`, `Interconnection`, `Pipe`), via `EdgeAsset`. Endpoints are `Reference[T]` objects holding the target's UUID.
- **`Collection`** — logical groupings (`Portfolio`, `Site`, `Region`, ...). Has `members` and `tz`. Not a graph vertex.
- **`Asset`** — cross-cutting mixin marking physical equipment (`commissioning_date`). Mixed into `NodeAsset` / `EdgeAsset`.

**Identity.** Every `Element` carries a `UUID7` generated at construction. The same UUID round-trips through JSON and (when paired with [EnergyDB](https://github.com/rebase-energy/energydb)) becomes the row primary key in PostgreSQL — no separate id field, no translation step.

This notebook walks through:

1. Building a portfolio tree declaratively.
2. Attaching time-series descriptors via the energy vocabulary.
3. Areas, edges, and cross-tree references.
4. Tree walking (`children()`, `to_tree()`).
5. Full JSON round-trip with UUID identity.


In [ ]:
import json

import energydatamodel as edm
from shapely.geometry import Polygon
from timedatamodel import DataType, Frequency

## 1. Build a tree

Build the whole portfolio top-down as one nested expression. Every `Node` takes a `members` list of child entities. Locations can be set with the `lat=` / `lon=` shorthand, or with a Shapely `geometry=Point(...)` for non-point shapes.

In [ ]:
portfolio = edm.Portfolio(
    name="Nordic",
    members=[
        edm.Site(
            name="Sweden-Offshore",
            lat=55.5,
            lon=12.8,
            members=[
                edm.wind.WindFarm(
                    name="Lillgrund",
                    capacity=110,
                    members=[
                        edm.wind.WindTurbine(name="T01", capacity=3.5, hub_height=80, lat=55.51, lon=12.78),
                        edm.wind.WindTurbine(name="T02", capacity=3.5, hub_height=80, lat=55.52, lon=12.79),
                    ],
                ),
                edm.solar.PVSystem(
                    name="Rooftop-01",
                    capacity=5.0,
                    surface_tilt=25,
                    surface_azimuth=180,
                    members=[
                        edm.solar.PVArray(capacity=5.0, surface_tilt=25, surface_azimuth=180),
                    ],
                ),
            ],
        ),
    ],
)
print(portfolio.to_tree())

## 2. Energy vocabulary

Time series are described by `TimeSeriesDescriptor`s built through convenience constructors. Each returns a descriptor whose `name` is a dotted string like `electricity.supply` or `price.spot`.

Under the hood `build_metric(Quantity, Kind, Scope)` composes these names; the constructors are thin wrappers.

In [ ]:
site = portfolio.members[0]
lillgrund = site.members[0]
rooftop = site.members[1]

lillgrund.members[0].timeseries = [
    edm.electricity_supply(unit="MW", data_type=DataType.FORECAST, frequency=Frequency.PT1H),
]
rooftop.timeseries = [
    edm.electricity_supply(unit="kW", data_type=DataType.ACTUAL),
]

lillgrund.members[0].timeseries[0]

In [ ]:
from energydatamodel import Kind, Quantity, Scope, build_metric

build_metric(Quantity.ELECTRICITY, Kind.DEMAND, Scope.AREA)

## 3. Areas + Edges + cross-tree references

`Area(Node)` represents bidding zones, control areas, countries, weather cells — each as a proper subclass (`BiddingZone`, `Country`, etc.), distinguished by `isinstance`.

`Interconnection(Edge)` connects two areas. The endpoint fields `from_element` / `to_element` are typed `Reference[T]`. A `Reference` captures another `Element`'s UUID:

- `Reference(other_element)` — captures `other_element.id`.
- `Reference(uuid_obj)` — by UUID directly.
- `Reference("01970000-...")` — by UUID string (handy for hand-edited JSON).

A bare `Element` passed to `Edge.from_element` / `to_element` is auto-wrapped in a `Reference` at construction. Resolution against a tree root (or a pre-built `Index`) is lazy and idempotent — refs are valid the moment they're created.

In [ ]:
se4 = edm.BiddingZone(name="SE4", timeseries=[edm.spot_price(unit="EUR / MWh")])
dk2 = edm.BiddingZone(name="DK2", timeseries=[edm.spot_price(unit="EUR / MWh")])
icx = edm.grid.Interconnection(
    name="SE4-DK2",
    from_element=edm.Reference(se4),  # captures se4.id
    to_element=edm.Reference(dk2),
    capacity_forward=1700,
    capacity_backward=1300,
    timeseries=[edm.cross_border_flow(unit="MW")],
)

portfolio = edm.Portfolio(
    name="Nordic",
    members=[site, se4, dk2, icx],
)
print(portfolio.to_tree())

# References hold UUIDs — both sides match.
print(f"\nicx.from_element.id == se4.id: {icx.from_element.id == se4.id}")

### Geospatial: points and polygons

Points are easiest via the `lat=` / `lon=` shorthand. For polygons (and other Shapely shapes), pass a `geometry=`. `Area`, `Region`, `SolarPowerArea`, `WindPowerArea` carry a `Polygon` (or `MultiPolygon`) in `geometry`. Derived values come straight from Shapely (`.area`, `.bounds`, `.centroid`). Geometry survives JSON round-trip via GeoJSON.

In [ ]:
# lat/lon shorthand on a point asset.
meter = edm.grid.Meter(name="M-1", lat=55.6, lon=13.0)
print("meter point:", (meter.longitude, meter.latitude))

# Polygons need an explicit geometry (lon, lat).
se4.geometry = Polygon([(12.5, 55.0), (16.5, 55.0), (16.5, 58.0), (12.5, 58.0)])
dk2.geometry = Polygon([(11.0, 54.5), (12.8, 54.5), (12.8, 56.0), (11.0, 56.0)])

print("SE4 area    :", se4.geometry.area)
print("SE4 bounds  :", se4.geometry.bounds)
print("SE4 centroid:", (se4.geometry.centroid.x, se4.geometry.centroid.y))

## 4. Tree walking and the Index

Every entity exposes a `children()` iterator. Recursive walks are trivial. For UUID lookups, `tree.index()` builds a `dict[UUID, Element]` via DFS — used internally by `Reference.resolve`, but also handy for ad-hoc queries.

In [ ]:
def walk(entity, depth=0):
    print("  " * depth + f"{type(entity).__name__}({entity.name!r})")
    for c in entity.children():
        walk(c, depth + 1)

walk(portfolio)

In [ ]:
# Index gives O(1) lookup by UUID.
index = portfolio.index()
print(f"index size: {len(index)}")

# Resolve the edge's endpoints via the index.
icx.from_element.resolve(index)
icx.to_element.resolve(index)
print(f"from_element → {icx.from_element.get().name}")
print(f"to_element   → {icx.to_element.get().name}")

## 5. JSON round-trip

`to_json()` emits a JSON-compatible dict. Every Element emits its `id` (UUID7) so round-trip preserves identity. Refs serialize as `{"__ref__": "<uuid>"}` — no path-walking required.

`from_json()` is **single-pass**: refs are valid the moment they're constructed. Resolution is on-demand via `Reference.resolve(root_or_index)`.

In [ ]:
js = portfolio.to_json()
print("Reference wire format:", next(m for m in js["members"] if m["__type__"] == "Interconnection")["from_element"])
# Each Element carries its `id` field on the wire.
print("Interconnection id:", next(m for m in js["members"] if m["__type__"] == "Interconnection")["id"])

In [ ]:
restored = edm.Portfolio.from_json(js)

# UUIDs round-trip — every Element keeps its identity.
assert restored.id == portfolio.id
assert restored.members[1].id == portfolio.members[1].id  # SE4

# Byte-equal after re-dump.
assert json.dumps(js, default=str) == json.dumps(restored.to_json(), default=str)

# Refs come back unresolved (single-pass deserialize); resolve on demand.
icx_restored = next(m for m in restored.members if isinstance(m, edm.grid.Interconnection))
print(f"resolved before resolve(): {icx_restored.from_element.is_resolved()}")
icx_restored.from_element.resolve(restored)
icx_restored.to_element.resolve(restored)
print(f"after resolve: from={icx_restored.from_element.get().name} to={icx_restored.to_element.get().name}")

### Error modes

A `Reference` whose UUID can't be found in the tree raises `UnresolvedReferenceError` at access time. The single-pass design means broken refs don't kill the load — they fail loudly only when you actually try to dereference them.

In [ ]:
from uuid import uuid4

from energydatamodel.reference import UnresolvedReferenceError

broken_ref = edm.Reference(uuid4())  # UUID that's not in any tree
try:
    broken_ref.resolve(portfolio)
except UnresolvedReferenceError as e:
    print("caught:", e)